<a href="https://colab.research.google.com/github/peremartra/CH13/blob/main/ch12_vllm_probe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 12 probe: does a model with attention-free blocks load in vLLM?

Minimal experiment to back the serving claims of Chapter 12. It targets `oopere/llama-3.2-3b-attn-drop-6` on a Colab **T4**.

What it does, in order:
1. Prints the exact library versions (the chapter must cite the vLLM version tested).
2. Inspects the checkpoint and builds a block inventory (which blocks keep attention).
3. Runs a Transformers baseline (correctness and batch-1 latency).
4. Tries to load the model in vLLM, as is. **The result of this step is the finding, whether it loads or not.**
5. If it loads: structured outputs, batch-1 latency and the KV cache memory vLLM actually allocates.

Run the cells in order. If the install cell changes the torch version, restart the runtime and run again from cell 2.

In [1]:
%pip -q install vllm huggingface_hub
import vllm, torch, transformers
print("vllm        :", vllm.__version__)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("GPU         :", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/7

In [2]:
MODEL_ID = "oopere/llama-3.2-3b-attn-drop-6"

# The T4 (compute capability 7.5) has no bfloat16 support, so we use float16 everywhere
DTYPE = torch.float16
BASE_BLOCKS = 28          # Llama-3.2-3B, used as the reference for ratios
BASE_KV_HEADS, HEAD_DIM = 8, 128

## 1. What is actually in the checkpoint

In [3]:
import json
from huggingface_hub import list_repo_files, hf_hub_download

# Reading the files first tells us whether the repo ships custom modeling code
print(list_repo_files(MODEL_ID))
config = json.load(open(hf_hub_download(MODEL_ID, "config.json")))
print(json.dumps(config, indent=2))

['.gitattributes', 'README.md', 'config.json', 'generation_config.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'modeling_pruned_llama.py', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']


config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

{
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoModelForCausalLM": "modeling_pruned_llama.PrunedLlamaForCausalLM"
  },
  "bos_token_id": 128000,
  "dropped_attn_layers": [
    18,
    20,
    21,
    22,
    23,
    24
  ],
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 24,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": true,
  "torch_dtype": "float16",
  "transformers_version": "4.55.4",
  "use_cache": true,
  "vocab_siz

## 2. Transformers baseline and block inventory

In [4]:
import gc, time, statistics
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map="cuda", trust_remote_code=True
).eval()

def attention_params(block):
    # We count parameters instead of checking a class name, because we don't know
    # how the removal was implemented (None, Identity or a no-op module)
    attn = getattr(block, "self_attn", None)
    return 0 if attn is None else sum(p.numel() for p in attn.parameters())

blocks = model.model.layers
no_attention = [i for i, b in enumerate(blocks) if attention_params(b) == 0]
inventory = {
    "blocks": len(blocks),
    "blocks_with_attention": len(blocks) - len(no_attention),
    "attention_free_indices": no_attention,
    "mlp_widths": sorted({b.mlp.gate_proj.out_features for b in blocks}),
    "params_billions": round(sum(p.numel() for p in model.parameters()) / 1e9, 3),
}
print(json.dumps(inventory, indent=2))

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/224 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

{
  "blocks": 28,
  "blocks_with_attention": 22,
  "attention_free_indices": [
    18,
    20,
    21,
    22,
    23,
    24
  ],
  "mlp_widths": [
    8192
  ],
  "params_billions": 3.062
}


In [5]:
# A plain few-shot prompt, because we only need a valid, parseable answer
# to compare engines, not to measure task accuracy
PROMPT = (
    "Convert the request into a JSON tool call.\n"
    "Request: look up the weather in Paris\n"
    'Call: {"name": "search", "arguments": {"query": "weather in Paris"}}\n'
    "Request: find the capital of Peru\n"
    "Call:"
)

inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")

def timed_generate(n_new=30):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=n_new, min_new_tokens=n_new, do_sample=False)
    torch.cuda.synchronize()
    return out, (time.perf_counter() - t0) * 1000 / n_new

timed_generate()                                      # warm-up, discarded
runs = [timed_generate()[1] for _ in range(10)]
out, _ = timed_generate()
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
tf_latency = {"p50_ms_per_token": round(statistics.median(runs), 1), "max_ms_per_token": round(max(runs), 1)}
print("Transformers, batch 1 (includes prefill):", tf_latency)

# vLLM needs the whole GPU, so we release this copy before the next step
del model; gc.collect(); torch.cuda.empty_cache()

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 {"name": "search", "arguments": {"query": "capital of Peru"}
Request: find the capital of Peru
Call: {"name":
Transformers, batch 1 (includes prefill): {'p50_ms_per_token': 39.1, 'max_ms_per_token': 50.2}


## 3. The test that decides the chapter: load it in vLLM

The probe runs in a **separate process**. If vLLM fails to load the model, it may leave the GPU in a bad state, and a subprocess keeps that failure from contaminating the rest of the notebook.

In [6]:
%%writefile vllm_probe.py
import json, sys, time, statistics, traceback

model_id, out_path = sys.argv[1], sys.argv[2]
result = {"loaded": False}

SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"const": "search"},
        "arguments": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    "required": ["name", "arguments"],
    "additionalProperties": False,
}
PROMPT = (
    "Convert the request into a JSON tool call.\n"
    "Request: look up the weather in Paris\n"
    'Call: {"name": "search", "arguments": {"query": "weather in Paris"}}\n'
    "Request: find the capital of Peru\n"
    "Call:"
)

try:
    import vllm
    from vllm import LLM, SamplingParams
    result["vllm_version"] = vllm.__version__

    # enforce_eager skips CUDA graph capture: fewer ways to fail on a T4,
    # at the price of somewhat pessimistic latency
    llm = LLM(model=model_id, dtype="half", trust_remote_code=True,
              max_model_len=2048, gpu_memory_utilization=0.85, enforce_eager=True)
    result["loaded"] = True

    # Free-form generation: same prompt as the Transformers baseline
    free = SamplingParams(temperature=0.0, max_tokens=30, ignore_eos=True)
    llm.generate([PROMPT], free)                          # warm-up
    times = []
    for _ in range(10):
        t0 = time.perf_counter()
        llm.generate([PROMPT], free)
        times.append((time.perf_counter() - t0) * 1000 / 30)
    result["free_text"] = llm.generate([PROMPT], free)[0].outputs[0].text
    result["latency"] = {"p50_ms_per_token": round(statistics.median(times), 1),
                         "max_ms_per_token": round(max(times), 1)}

    # Structured outputs: the parameter name changed between vLLM releases,
    # so we try the new API first and fall back to the old one
    try:
        from vllm.sampling_params import StructuredOutputsParams
        guided = SamplingParams(temperature=0.0, max_tokens=64,
                                structured_outputs=StructuredOutputsParams(json=SCHEMA))
        result["structured_api"] = "structured_outputs"
    except Exception:
        from vllm.sampling_params import GuidedDecodingParams
        guided = SamplingParams(temperature=0.0, max_tokens=64,
                                guided_decoding=GuidedDecodingParams(json=SCHEMA))
        result["structured_api"] = "guided_decoding"
    text = llm.generate([PROMPT], guided)[0].outputs[0].text
    result["structured_text"] = text
    try:
        json.loads(text); result["structured_valid_json"] = True
    except Exception:
        result["structured_valid_json"] = False
except Exception:
    result["error"] = traceback.format_exc()[-3000:]

json.dump(result, open(out_path, "w"), indent=2)

Writing vllm_probe.py


In [7]:
import os, re, subprocess

# Some vLLM versions need the older engine on a T4; we try the default first
# and only then the fallback, and keep every error because failures are data too
ATTEMPTS = [{"label": "default engine", "env": {}},
            {"label": "VLLM_USE_V1=0", "env": {"VLLM_USE_V1": "0"}}]

attempts_log, vllm_result, log_text = [], None, ""
for attempt in ATTEMPTS:
    path = f"vllm_result_{len(attempts_log)}.json"
    proc = subprocess.run(["python", "vllm_probe.py", MODEL_ID, path],
                          env={**os.environ, **attempt["env"]},
                          capture_output=True, text=True)
    log_text = proc.stdout + proc.stderr
    res = json.load(open(path)) if os.path.exists(path) else {"loaded": False, "error": log_text[-3000:]}
    attempts_log.append((attempt["label"], res))
    if res["loaded"]:
        vllm_result = res
        break

for label, res in attempts_log:
    print(f"[{label}] loaded={res['loaded']}")
    if not res["loaded"]:
        print(res.get("error", "")[-1500:], "\n")

[default engine] loaded=False
t.py", line 122, in make_client
    return SyncMPClient(
        vllm_config,
    ...<2 lines>...
        renderer=renderer,
    )
  File "/usr/local/lib/python3.13/dist-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/core_client.py", line 880, in __init__
    super().__init__(
    ~~~~~~~~~~~~~~~~^
        asyncio_mode=False,
        ^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
        renderer=renderer,
        ^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/core_client.py", line 656, in __init__
    with launch_core_engines(
         ~~~~~~~~~~~~~~~~~~~^
        vllm_config, executor_class, log_stats, addresses
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ) as engine_launch:
    ^
  File "/usr/lib/python3.13/contextlib.py", line 148, in __exit__
    next(self.gen)
    ~~~~^^^^^^^^^^
  File "/

## 4. Reading the result

In [8]:
if vllm_result is None:
    print("vLLM did NOT load the model as is. Section 12.6 should say so, with the error above and the vLLM version.")
else:
    print("vLLM version        :", vllm_result["vllm_version"])
    print("Structured API      :", vllm_result["structured_api"])
    print("Structured valid    :", vllm_result["structured_valid_json"], "->", vllm_result["structured_text"])
    print("Free-form output    :", vllm_result["free_text"])
    print("Transformers latency:", tf_latency)
    print("vLLM latency        :", vllm_result["latency"])

    # vLLM logs how much memory it reserves for the KV cache and how many tokens fit.
    # Dividing one by the other gives the bytes it really allocates per token,
    # which we can compare with what the formula predicts for this block inventory
    mem = re.search(r"Available KV cache memory: ([\d.]+) GiB", log_text)
    tok = re.search(r"GPU KV cache size: ([\d,]+) tokens", log_text)
    if mem and tok:
        measured = float(mem.group(1)) * 2**30 / int(tok.group(1).replace(",", ""))
        per_layer = 2 * BASE_KV_HEADS * HEAD_DIM * 2                     # K+V, float16
        l_attn = inventory["blocks_with_attention"]
        print(f"Measured bytes/token : {measured:,.0f}")
        print(f"Formula, {l_attn} attention blocks: {per_layer * l_attn:,}")
        print(f"Formula, {inventory['blocks']} blocks (uniform allocation): {per_layer * inventory['blocks']:,}")
    else:
        print("KV cache lines not found in the log; copy them by hand from the vLLM output.")

vLLM did NOT load the model as is. Section 12.6 should say so, with the error above and the vLLM version.


### How to read the KV cache comparison

- Measured bytes/token close to the **attention-blocks** formula: vLLM only reserves cache for blocks that have attention.
- Measured close to the **all-blocks** formula: vLLM allocates uniformly and the saving does not materialize. This is the "heterogeneity catch" of section 12.7, now measured.

Both outcomes are publishable. Report the vLLM version and the date next to the result.